In [2]:
import warnings

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(split_name: str, feature_name: str) -> pd.DataFrame:
    path = f"datasets/{split_name}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(split_name: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(split_name, feature_names[0])
    for name in feature_names[1:]:
        extra = load_split(split_name, name)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    dtrain = lgb.Dataset(train_df[feat_cols], label=np.log1p(train_df["target"].values))
    dval = lgb.Dataset(val_df[feat_cols], label=np.log1p(val_df["target"].values), reference=dtrain)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 63,  # ~2^depth for depth=6
        "max_depth": 6,
        "verbosity": -1,
        "seed": 42,
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=-1),
    ]

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=callbacks,
    )

    preds = np.expm1(model.predict(val_df[feat_cols].values))
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split("split1", feature_name)
    val1_df = load_split("split2", feature_name)
    val2_df = load_split("split3", feature_name)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(
        f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": "split1_" + feature_name, "groups": feature_name, "rmsle": score1})
    results.append({"experiment": "split2_" + feature_name, "groups": feature_name, "rmsle": score2})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("split1", subset)
    val1_df = merge_splits("split2", subset)
    val2_df = merge_splits("split3", subset)
    score1, best_iter1 = train_and_eval(train_df, val1_df)
    score2, best_iter2 = train_and_eval(train_df, val2_df)
    print(
        f" split1 RMSLE = {score1:.5f}  (best iteration: {best_iter1}) | split2 RMSLE = {score2:.5f}  (best iteration: {best_iter2})")
    results.append({"experiment": f"split1_cumulative_{i}", "groups": label, "rmsle": score1})
    results.append({"experiment": f"split2_cumulative_{i}", "groups": label, "rmsle": score2})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))

 split1 RMSLE = 1.74572  (best iteration: 113) | split2 RMSLE = 1.69846  (best iteration: 195)

Ranking by RMSLE (lower = better)
             experiment                                                                                 groups    rmsle
    split2_cumulative_6                          base + all_history + rolling_windows + velocity + trend + gmv 1.698444
    split2_cumulative_8 base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 1.698459
    split2_cumulative_7       base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 1.698476
    split2_cumulative_5                                base + all_history + rolling_windows + velocity + trend 1.698773
    split2_cumulative_3                                                   base + all_history + rolling_windows 1.699168
    split2_cumulative_4                                        base + all_history + rolling_windows + velocity 1.699186
        split2_velocity       

In [6]:
import joblib
import lightgbm as lgb
import optuna

RANDOM_STATE = 42

lgb_train = pd.read_parquet("datasets/split1_all_features.parquet")
lgb_val = pd.read_parquet("datasets/split2_all_features.parquet")
lgb_val2 = pd.read_parquet("datasets/split3_all_features.parquet")

lgb_feat_cols = [c for c in lgb_train.columns if
                 c not in ("user_id", "target") and not
                 pd.api.types.is_datetime64_any_dtype(lgb_train[c])]

print(f"  LightGBM  — train: {lgb_train.shape[0]} rows, {len(lgb_feat_cols)} features")

X_lgb_tr = lgb_train[lgb_feat_cols].values
y_lgb_tr = np.log1p(lgb_train["target"].values)
X_lgb_val = lgb_val[lgb_feat_cols].values
y_lgb_val = np.log1p(lgb_val["target"].values)
X_lgb_val2 = lgb_val2[lgb_feat_cols].values
y_lgb_val2 = np.log1p(lgb_val2["target"].values)
y_lgb_val_raw = lgb_val["target"].values
y_lgb_val2_raw = lgb_val2["target"].values

print("\nOptuna tuning LightGBM (20 trials)...")


def lgb_objective(trial):
    params = dict(
        objective="regression",
        metric="rmse",
        n_estimators=trial.suggest_int("n_estimators", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_lgb_tr, y_lgb_tr,
        eval_set=[(X_lgb_val, y_lgb_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    score1 = rmsle(y_lgb_val_raw, np.expm1(model.predict(X_lgb_val)))
    score2 = rmsle(y_lgb_val2_raw, np.expm1(model.predict(X_lgb_val2)))
    return (score1 + score2) / 2


lgb_study = optuna.create_study(direction="minimize")
lgb_study.optimize(lgb_objective, n_trials=20)
best_lgb_params = lgb_study.best_params

print(f"  Best LightGBM RMSLE: {lgb_study.best_value:.5f}")
print(f"  Params: {best_lgb_params}")

final_lgb = lgb.LGBMRegressor(
    **best_lgb_params,
    objective="regression",
    metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
final_lgb.fit(
    X_lgb_tr, y_lgb_tr,
    eval_set=[(X_lgb_val, y_lgb_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

lgb_best_n = final_lgb.best_iteration_

X_lgb_full = np.concatenate([X_lgb_tr, X_lgb_val])
y_lgb_full = np.concatenate([y_lgb_tr, y_lgb_val])

lgb_sub_params = {k: v for k, v in best_lgb_params.items() if k != "n_estimators"}

final_lgb_full = lgb.LGBMRegressor(
    **lgb_sub_params,
    n_estimators=lgb_best_n,
    objective="regression",
    metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
final_lgb_full.fit(X_lgb_full, y_lgb_full)

joblib.dump(final_lgb, "model/lgb_split1.joblib")
joblib.dump(final_lgb_full, "model/lgb_full.joblib")

['model/lgb_full.joblib']

In [3]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

FEATURES_GROUP = [
    "base",
    "all_history",
    "rolling_windows",
    "velocity",
    "trend",
    "gmv",
    "conversion_rates",
    "ewm",
]


def rmsle(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_pred = np.maximum(y_pred, 0)
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


def load_split(feature_name: str, t: str) -> pd.DataFrame:
    path = f"datasets/{t}_{feature_name}.parquet"
    return pd.read_parquet(path)


def get_feature_cols(df: pd.DataFrame) -> list[str]:
    exclude = {"user_id", "target"}
    return [
        c for c in df.columns
        if c not in exclude and not pd.api.types.is_datetime64_any_dtype(df[c])
    ]


def merge_splits(t: str, feature_names: list[str]) -> pd.DataFrame:
    base = load_split(feature_names[0], t)
    for name in feature_names[1:]:
        extra = load_split(name, t)
        new_cols = [c for c in extra.columns if c not in base.columns]
        base = base.merge(extra[["user_id"] + new_cols], on="user_id", how="left")
    return base


def train_and_eval(train_df: pd.DataFrame, val_df: pd.DataFrame) -> tuple[float, int]:
    feat_cols = get_feature_cols(train_df)

    train_y = np.log1p(train_df["target"].values)
    val_y = np.log1p(val_df["target"].values)

    dtrain = lgb.Dataset(train_df[feat_cols].values, label=train_y)
    dval = lgb.Dataset(val_df[feat_cols].values, label=val_y, reference=dtrain)

    params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 63,
        "verbose": -1,
        "seed": 42,
    }

    callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=False)]

    model = lgb.train(
        params,
        dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=callbacks,
    )

    preds = np.expm1(model.predict(val_df[feat_cols].values))
    score = rmsle(val_df["target"].values, preds)
    return score, model.best_iteration


# =============================================================================
# Main
# =============================================================================

results = []

print("\n--- Individual feature groups ---")
for feature_name in FEATURES_GROUP:
    print(f"\nFeature group: {feature_name}")
    train_df = load_split(feature_name, "train")
    val_df = load_split(feature_name, "test")
    score, best_iter = train_and_eval(train_df, val_df)
    print(f" RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": feature_name, "rmsle": score})

print("\n--- Cumulative feature groups ---")
for i in range(2, len(FEATURES_GROUP) + 1):
    subset = FEATURES_GROUP[:i]
    label = " + ".join(subset)
    print(f"\nCumulative: {label}")
    train_df = merge_splits("train", subset)
    val_df = merge_splits("test", subset)
    score, best_iter = train_and_eval(train_df, val_df)
    print(f" RMSLE = {score:.5f}  (best iteration: {best_iter})")
    results.append({"experiment": label, "rmsle": score})

# =============================================================================
# Summary
# =============================================================================

results_df = pd.DataFrame(results).sort_values("rmsle")

print("\n" + "=" * 80)
print("Ranking by RMSLE (lower = better)")
print("=" * 80)
print(results_df.to_string(index=False))


 RMSLE = 1.67153  (best iteration: 151)

Ranking by RMSLE (lower = better)
                                                                            experiment    rmsle
base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates + ewm 1.671535
                         base + all_history + rolling_windows + velocity + trend + gmv 1.671756
                                       base + all_history + rolling_windows + velocity 1.671900
                                                                                 trend 1.672192
      base + all_history + rolling_windows + velocity + trend + gmv + conversion_rates 1.672296
                                                  base + all_history + rolling_windows 1.672541
                               base + all_history + rolling_windows + velocity + trend 1.672666
                                                                              velocity 1.672905
                                                             

In [5]:
import joblib
import lightgbm as lgb
import optuna

RANDOM_STATE = 42

lgb_train = pd.read_parquet("datasets/train_all_features.parquet")
lgb_val = pd.read_parquet("datasets/test_all_features.parquet")

lgb_feat_cols = [c for c in lgb_train.columns if
                 c not in ("user_id", "target") and not
                 pd.api.types.is_datetime64_any_dtype(lgb_train[c])]

print(f"  LightGBM  — train: {lgb_train.shape[0]} rows, {len(lgb_feat_cols)} features")

X_lgb_tr = lgb_train[lgb_feat_cols].values
y_lgb_tr = np.log1p(lgb_train["target"].values)
X_lgb_val = lgb_val[lgb_feat_cols].values
y_lgb_val = np.log1p(lgb_val["target"].values)
y_lgb_val_raw = lgb_val["target"].values

print("\nOptuna tuning LightGBM (20 trials)...")


def lgb_objective(trial):
    params = dict(
        objective="regression",
        metric="rmse",
        n_estimators=trial.suggest_int("n_estimators", 500, 3000),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_lgb_tr, y_lgb_tr,
        eval_set=[(X_lgb_val, y_lgb_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    score = rmsle(y_lgb_val_raw, np.expm1(model.predict(X_lgb_val)))
    return score


lgb_study = optuna.create_study(direction="minimize")
lgb_study.optimize(lgb_objective, n_trials=20)
best_lgb_params = lgb_study.best_params

print(f"  Best LightGBM RMSLE: {lgb_study.best_value:.5f}")
print(f"  Params: {best_lgb_params}")

final_lgb = lgb.LGBMRegressor(
    **best_lgb_params,
    objective="regression",
    metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
final_lgb.fit(
    X_lgb_tr, y_lgb_tr,
    eval_set=[(X_lgb_val, y_lgb_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
)

lgb_best_n = final_lgb.best_iteration_

X_lgb_full = np.concatenate([X_lgb_tr, X_lgb_val])
y_lgb_full = np.concatenate([y_lgb_tr, y_lgb_val])

lgb_sub_params = {k: v for k, v in best_lgb_params.items() if k != "n_estimators"}

final_lgb_full = lgb.LGBMRegressor(
    **lgb_sub_params,
    n_estimators=lgb_best_n,
    objective="regression",
    metric="rmse",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
final_lgb_full.fit(X_lgb_full, y_lgb_full)

joblib.dump(final_lgb, "model/lgb_split1.joblib")
joblib.dump(final_lgb_full, "model/lgb_full.joblib")

['model/lgb_full.joblib']